In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [4]:
ARQUIVO_RAW = "../data/raw/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

df_22_raw = pd.read_excel(ARQUIVO_RAW, sheet_name="PEDE2022")
df_23_raw = pd.read_excel(ARQUIVO_RAW, sheet_name="PEDE2023")
df_24_raw = pd.read_excel(ARQUIVO_RAW, sheet_name="PEDE2024")

# Cópias para diagnóstico/EDA
df_22_eda = df_22_raw.copy()
df_23_eda = df_23_raw.copy()
df_24_eda = df_24_raw.copy()

## Diagnóstico inicial de schema

In [3]:
bases_eda = {
    "2022": df_22_eda,
    "2023": df_23_eda,
    "2024": df_24_eda
}

# Quantidade de linhas e colunas por base
resumo_shape = pd.DataFrame([
    {"ano": ano, "linhas": df.shape[0], "colunas": df.shape[1]}
    for ano, df in bases_eda.items()
])

resumo_shape

,ano,linhas,colunas
0,2022,860,42
1,2023,1014,48
2,2024,1156,50


Antes das análises, fizemos uma etapa de diagnóstico da base. Um ponto importante foi que a base disponibilizada para o Datathon não necessariamente representa todo o universo citado nos relatórios PEDE. Por exemplo, enquanto o relatório PEDE 2022 apresenta 929 estudantes em alguns recortes, a base de 2022 usada no desafio possui 860 registros, número que também aparece em outros recortes do próprio relatório. Por isso, usamos os relatórios como referência metodológica, mas limitamos as conclusões ao recorte efetivamente disponibilizado

In [4]:
cols_22 = set(df_22_eda.columns)
cols_23 = set(df_23_eda.columns)
cols_24 = set(df_24_eda.columns)

print("Colunas somente em 2022:")
print(sorted(cols_22 - cols_23 - cols_24))

print("\nColunas somente em 2023:")
print(sorted(cols_23 - cols_22 - cols_24))

print("\nColunas somente em 2024:")
print(sorted(cols_24 - cols_22 - cols_23))

print("\nColunas comuns aos três anos:")
print(sorted(cols_22 & cols_23 & cols_24))

Colunas somente em 2022:
['Ano nasc', 'Defas', 'Fase ideal', 'Idade 22', 'Inglês', 'Matem', 'Nome', 'Portug']

Colunas somente em 2023:
['Destaque IPV.1', 'INDE 2023', 'Pedra 2023']

Colunas somente em 2024:
['Ativo/ Inativo', 'Ativo/ Inativo.1', 'Avaliador5', 'Avaliador6', 'Escola', 'INDE 2024', 'Pedra 2024']

Colunas comuns aos três anos:
['Ano ingresso', 'Atingiu PV', 'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4', 'Cf', 'Cg', 'Ct', 'Destaque IDA', 'Destaque IEG', 'Destaque IPV', 'Fase', 'Gênero', 'IAA', 'IAN', 'IDA', 'IEG', 'INDE 22', 'IPS', 'IPV', 'Indicado', 'Instituição de ensino', 'Nº Av', 'Pedra 20', 'Pedra 21', 'Pedra 22', 'RA', 'Rec Av1', 'Rec Av2', 'Rec Psicologia', 'Turma']


# Identificado ausencia da coluna IPP (indicador psicopedagogico) na base de 2022

In [5]:
schema_resumo = pd.DataFrame({
    "coluna_2022": pd.Series(df_22_eda.columns),
    "coluna_2023": pd.Series(df_23_eda.columns),
    "coluna_2024": pd.Series(df_24_eda.columns)
})

schema_resumo

,coluna_2022,coluna_2023,coluna_2024
0,RA,RA,RA
1,Fase,Fase,Fase
2,Turma,INDE 2023,INDE 2024
3,Nome,Pedra 2023,Pedra 2024
4,Ano nasc,Turma,Turma
5,Idade 22,Nome Anonimizado,Nome Anonimizado
6,Gênero,Data de Nasc,Data de Nasc
7,Ano ingresso,Idade,Idade
8,Instituição de ensino,Gênero,Gênero
9,Pedra 20,Ano ingresso,Ano ingresso


## Diagnóstico de nulos e erros de fórmula

In [6]:
def percentual_nulos(df, nome_base, limiar=0):
    nulos_pct = (df.isnull().sum() / len(df)) * 100
    nulos_pct = nulos_pct[nulos_pct > limiar].sort_values(ascending=False)

    return pd.DataFrame({
        "base": nome_base,
        "coluna": nulos_pct.index,
        "perc_nulos": nulos_pct.round(2).values
    })

nulos_22 = percentual_nulos(df_22_eda, "2022", limiar=0)
nulos_23 = percentual_nulos(df_23_eda, "2023", limiar=0)
nulos_24 = percentual_nulos(df_24_eda, "2024", limiar=0)

diag_nulos = pd.concat([nulos_22, nulos_23, nulos_24], ignore_index=True)
diag_nulos.sort_values(["base", "perc_nulos"], ascending=[True, False])

,base,coluna,perc_nulos
0,2022,Inglês,67.09
1,2022,Rec Av4,65.58
2,2022,Avaliador4,63.95
3,2022,Pedra 20,62.44
4,2022,Pedra 21,46.28
...,...,...,...
74,2024,IDA,8.74
75,2024,INDE 2024,5.54
76,2024,Pedra 2024,5.54
77,2024,Instituição de ensino,0.09


In [7]:
erros_excel = [
    "#N/D",
    "#N/A",
    "#DIV/0!",
    "#VALUE!",
    "#REF!",
    "#NUM!",
    "#NAME?",
    "#NULL!",
    "INCLUIR",
    ""
]

def diagnosticar_erros_excel(df, nome_base):
    resultados = []

    for col in df.columns:
        serie = df[col].astype(str).str.strip().str.upper()

        for erro in erros_excel:
            qtd = (serie == erro.upper()).sum()

            if qtd > 0:
                resultados.append({
                    "base": nome_base,
                    "coluna": col,
                    "erro": erro,
                    "qtd_erros": int(qtd),
                    "perc_erros": round((qtd / len(df)) * 100, 2)
                })

    return pd.DataFrame(resultados)

diag_erros = pd.concat([
    diagnosticar_erros_excel(df_22_eda, "2022"),
    diagnosticar_erros_excel(df_23_eda, "2023"),
    diagnosticar_erros_excel(df_24_eda, "2024")
], ignore_index=True)

diag_erros.sort_values(["base", "qtd_erros"], ascending=[True, False])

,base,coluna,erro,qtd_erros,perc_erros
0,2024,INDE 2024,INCLUIR,38,3.29
1,2024,Pedra 2024,INCLUIR,38,3.29


**Observação metodológica:** valores como `#N/D` e `#DIV/0!` podem ser interpretados pelo `pandas` como nulos durante a leitura do Excel. Por isso, o diagnóstico de nulos e o diagnóstico de erros textuais devem ser analisados em conjunto.

## Diagnóstico de colunas-chave

In [8]:
# Coluna fase possui padroes de escrita diferente entre as bases 
for ano, df in bases_eda.items():
    print(f"\nValores únicos - Fase {ano}")
    print(df["Fase"].dropna().unique())


Valores únicos - Fase 2022
[7 6 5 4 3 2 1 0]

Valores únicos - Fase 2023
['ALFA' 'FASE 1' 'FASE 2' 'FASE 3' 'FASE 4' 'FASE 5' 'FASE 6' 'FASE 7'
 'FASE 8']

Valores únicos - Fase 2024
['ALFA' '1A' '1B' '1C' '1D' '1E' '1G' '1H' '1J' '1K' '1L' '1M' '1N' '1P'
 '1R' '2A' '2B' '2C' '2D' '2G' '2H' '2I' '2K' '2L' '2M' '2N' '2P' '2R'
 '2U' '3A' '3B' '3C' '3D' '3F' '3G' '3H' '3I' '3K' '3L' '3M' '3N' '3P'
 '3R' '3U' '4A' '4B' '4C' '4F' '4H' '4L' '4M' '4N' '4R' '5A' '5B' '5C'
 '5D' '5F' '5G' '5L' '5M' '5N' '6A' '6L' '7A' '7E' '8A' '8B' '8D' '8E'
 '8F' 9]


In [9]:
# Instituição de ensino possui padroes de escrita diferente entre as bases
for ano, df in bases_eda.items():
    print(f"\nValores únicos - Instituição de ensino {ano}")
    print(df["Instituição de ensino"].dropna().unique())


Valores únicos - Instituição de ensino 2022
['Escola Pública' 'Rede Decisão' 'Escola JP II']

Valores únicos - Instituição de ensino 2023
['Pública' 'Privada' 'Privada - Programa de Apadrinhamento'
 'Privada - Programa de apadrinhamento' 'Concluiu o 3º EM'
 'Nenhuma das opções acima' 'Privada *Parcerias com Bolsa 100%'
 'Privada - Pagamento por *Empresa Parceira']

Valores únicos - Instituição de ensino 2024
['Pública' 'Privada' 'Privada - Programa de apadrinhamento'
 'Privada - Programa de Apadrinhamento' 'Concluiu o 3º EM'
 'Privada *Parcerias com Bolsa 100%'
 'Privada - Pagamento por *Empresa Parceira'
 'Bolsista Universitário *Formado (a)']


In [5]:
print("Valores únicos - Gênero 2022")
print(df_22_eda['Gênero'].value_counts(dropna=False))

print("\nValores únicos - Gênero 2023")
print(df_23_eda['Gênero'].value_counts(dropna=False))

print("\nValores únicos - Gênero 2024")
print(df_24_eda['Gênero'].value_counts(dropna=False))

Valores únicos - Gênero 2022
Gênero
Menina    457
Menino    403
Name: count, dtype: int64

Valores únicos - Gênero 2023
Gênero
Feminino     546
Masculino    468
Name: count, dtype: int64

Valores únicos - Gênero 2024
Gênero
Feminino     623
Masculino    533
Name: count, dtype: int64


## Diagnóstico da variável Gênero

Identificado inconsistência na variável `Gênero` entre os anos. Valores `Menina` e `Menino`; em outras, como `Feminino` e `Masculino`. Aplicado padronização para `Feminino` e `Masculino`, mantendo a comparabilidade longitudinal entre 2022, 2023 e 2024.

In [10]:
# Fase ideal
print("Valores únicos - Fase ideal 2022")
print(df_22_eda["Fase ideal"].dropna().unique())

print("\nValores únicos - Fase ideal 2023")
print(df_23_eda["Fase Ideal"].dropna().unique())

print("\nValores únicos - Fase ideal 2024")
print(df_24_eda["Fase Ideal"].dropna().unique())

Valores únicos - Fase ideal 2022
['Fase 8 (Universitários)' 'Fase 7 (3º EM)' 'Fase 6 (2º EM)'
 'Fase 5 (1º EM)' 'Fase 3 (7º e 8º ano)' 'Fase 4 (9º ano)'
 'Fase 2 (5º e 6º ano)' 'Fase 1 (4º ano)' 'ALFA  (2º e 3º ano)']

Valores únicos - Fase ideal 2023
['ALFA (1° e 2° ano)' 'Fase 1 (3° e 4° ano)' 'Fase 2 (5° e 6° ano)'
 'Fase 3 (7° e 8° ano)' 'Fase 4 (9° ano)' 'Fase 5 (1° EM)'
 'Fase 6 (2° EM)' 'Fase 7 (3° EM)' 'Fase 8 (Universitários)']

Valores únicos - Fase ideal 2024
['ALFA (1° e 2° ano)' 'Fase 1 (3° e 4° ano)' 'Fase 2 (5° e 6° ano)'
 'Fase 3 (7° e 8° ano)' 'Fase 4 (9° ano)' 'Fase 6 (2° EM)'
 'Fase 5 (1° EM)' 'Fase 7 (3° EM)' 'Fase 8 (Universitários)']


## Validação de idade

In [11]:
# 2022
idade_22 = pd.to_numeric(df_22_eda["Idade 22"], errors="coerce")
ano_nasc_22 = pd.to_numeric(df_22_eda["Ano nasc"], errors="coerce")
idade_calculada_22 = 2022 - ano_nasc_22

print("Validação idade 2022")
print((idade_calculada_22 == idade_22).value_counts(dropna=False))

# 2023
data_nasc_23 = pd.to_datetime(df_23_eda["Data de Nasc"], errors="coerce", dayfirst=True)
idade_23 = pd.to_numeric(df_23_eda["Idade"], errors="coerce")
idade_calculada_23 = 2023 - data_nasc_23.dt.year

print("\nValidação idade 2023")
print((idade_calculada_23 == idade_23).value_counts(dropna=False))
print("Diferença idade 2023")
print((idade_calculada_23 - idade_23).value_counts(dropna=False).sort_index())

# 2024
data_nasc_24 = pd.to_datetime(df_24_eda["Data de Nasc"], errors="coerce", dayfirst=True)
idade_24 = pd.to_numeric(df_24_eda["Idade"], errors="coerce")
idade_calculada_24 = 2024 - data_nasc_24.dt.year

print("\nValidação idade 2024")
print((idade_calculada_24 == idade_24).value_counts(dropna=False))
print("Diferença idade 2024")
print((idade_calculada_24 - idade_24).value_counts(dropna=False).sort_index())

Validação idade 2022
True    860
Name: count, dtype: int64

Validação idade 2023
False    522
True     492
Name: count, dtype: int64
Diferença idade 2023
0.0    492
1.0    123
NaN    399
Name: count, dtype: int64

Validação idade 2024
True     982
False    174
Name: count, dtype: int64
Diferença idade 2024
0    982
1    174
Name: count, dtype: int64


C:\Users\andre\AppData\Local\Temp\ipykernel_23272\2589863434.py:10: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  data_nasc_23 = pd.to_datetime(df_23_eda["Data de Nasc"], errors="coerce", dayfirst=True)


Na validação da idade, a base de 2022 apresentou consistência integral entre idade informada e idade calculada pelo ano de referência. Nas bases posteriores, as divergências observadas ficaram concentradas em diferença de um ano, sugerindo diferença de critério entre idade na data de coleta e idade pelo ano civil. Para garantir comparabilidade longitudinal, foi adotada a idade calculada por `ano_ref - ano_nascimento`, preservando flags de padronização na camada silver.

## Validação da chave do aluno

In [12]:
def validar_chave_ra_nome(df, col_ra, col_nome):
    temp = df.copy()
    temp["id_ra"] = temp[col_ra].astype(str).str.extract(r"(\d+)$")
    temp["id_nome"] = temp[col_nome].astype(str).str.extract(r"(\d+)$")
    temp["chave_ok"] = temp["id_ra"] == temp["id_nome"]
    return temp["chave_ok"].value_counts(dropna=False)

print("2022")
print(validar_chave_ra_nome(df_22_eda, "RA", "Nome"))

print("\n2023")
print(validar_chave_ra_nome(df_23_eda, "RA", "Nome Anonimizado"))

print("\n2024")
print(validar_chave_ra_nome(df_24_eda, "RA", "Nome Anonimizado"))

2022
chave_ok
True    860
Name: count, dtype: int64

2023
chave_ok
True    1014
Name: count, dtype: int64

2024
chave_ok
True    1156
Name: count, dtype: int64


Identificado duplicidade de colunas entre RA e NOME ANONIMIZADO. Desta forma decidiu-se criar uma chave primaria geral ID

Registros classificados como Fase 9 foram identificados como casos fora da estrutura metodológica principal do PEDE, que considera fases de Alfa/0 a 8. Na camada silver, esses registros foram preservados com uma flag de rastreabilidade. Para as análises do DATATHON e modelagem preditiva, optou-se por excluí-los, evitando mistura entre alunos regulares e casos operacionais/universitários/formados.

## Decisões de remoção de colunas

Algumas colunas foram removidas da base analítica por não atenderem aos critérios de comparabilidade longitudinal, aderência às perguntas norteadoras ou relevância para modelagem.

- `Destaque IEG`, `Destaque IDA`, `Destaque IPV`: informações textuais presentes de forma inconsistente entre os anos, não utilizadas na modelagem inicial.
- `Avaliador1`, `Avaliador2`, `Rec Av1` etc.: colunas operacionais de avaliação sem continuidade suficiente entre 2022, 2023 e 2024.
- `Turma`: variável operacional de alocação, sem relação direta com as perguntas norteadoras.
- `Cg`, `Cf`, `Ct`: rankings disponíveis apenas em 2022, sem continuidade longitudinal.
- `Rec Psicologia`, `Indicado`: campos não comparáveis entre os anos.